# **Build LangGraph Design Patterns: Orchestration**

# Orchestrator-Worker Pattern

The orchestrator-worker pattern is a powerful approach for handling complex, unpredictable tasks by dynamically **breaking them into manageable pieces** and **processing them in parallel**. When you can't predict upfront how many subtasks you'll need or what they'll involve, this pattern shines by intelligently **analyzing** the input, **decomposing** it into structured work units, and **assigning** specialized workers to handle each piece independently. This pattern is widely used in real-world AI workflow systems, including industry solutions such as IBM’s [Watsonx Orchestrate](https://www.ibm.com/products/watsonx-orchestrate?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-Agentic+Design+Patterns+in+LangGraph-v1_1752612018).

Think of it like a busy catering kitchen that takes orders for large events—when a customer orders "I want a three-course meal, hamburgers for lunch, and pizza for dinner," the head chef (orchestrator) analyzes the request and breaks it down into specific dishes, then assigns specialized culinary consultants to create detailed meal plans for each cuisine. Each consultant specializes in a specific cuisine, for example, one expert in Italian cuisine handles the pizza planning, while an American cuisine specialist develops the hamburger menu. The key is that each consultant receives a **clear, structured brief** with dish requirements and dietary considerations, not just "plan something good." In our meal planning system, user requests such as "feed my family for the week" could result in anywhere from 3 to 15 different dishes, each requiring a specialist consultant's expertise in their particular cuisine to create comprehensive meal plans.

Here's a simple visualization of the workflow (made on [Excalidraw](https://excalidraw.com/)):

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XKXeMzLe9KNW5UjYiJpuSQ/orchestration.png" width="100%" alt="orchestration">

## Structured Output

The orchestrator **must** produce **structured outputs** because our worker nodes require specific, well-defined input and output formats to process information reliably. Each worker (chef consultant) specializes in gathering specific information about their assigned cuisine:

- Name of the dish (for example, "Margherita Pizza", "Classic Cheeseburger")
- List of ingredients (the most important part for meal planning)
- The cuisine or cultural origin of the dish (for example, Italian, American, Mexican)

For each worker, we aggregate this data into a structured format called a `Dish`. We can then store a list of dishes with the `Dishes` class.


In [1]:
from langchain_groq import ChatGroq
import os
import dotenv

d:\Projects\langgraph-explore\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dotenv.load_dotenv()
api = os.getenv("groq_api")

In [3]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api,
    temperature=0.5
)

### Dish Class

In [4]:
from pydantic import BaseModel, Field

In [5]:
class Dish(BaseModel):
    
    name:str = Field(description="Name of the dish (for example, Spaghetti Bolognese, Chicken Curry).")
    
    ingredients: list[str] = Field(description="List of ingredients needed for this dish, sperated by comma")
    
    location:str = Field(description="The cuisine or cultural origin of the dish (for example, Italian, Indian, Mexican).")

In [6]:
class Dishes(BaseModel):
    
    sections: list[Dish] = Field(description="A list of grocery sections, one for each dish, with ingredients.")

In [7]:
from langchain_core.prompts import ChatPromptTemplate

In [8]:
dish_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an assistant that generates a structured grocery list.\n\n"
            "The user wants to prepare the following meal: {meals}\n\n"
            "for each meal, return a seaction with:\n"
            "- the name of the dish\n"
            "- a comma-seperated list of ingredients needed for that dish.\n"
            "- the cuisine or cultural origin of the food."
        )
    ]
)

In [9]:
planner_pipe = dish_prompt | llm.with_structured_output(Dishes)

In [10]:
planner_pipe.invoke({
    "meals": ["banana smoothie", "carrot cake"]
})

Dishes(sections=[Dish(name='banana smoothie', ingredients=['banana', 'yogurt', 'milk', 'honey'], location='American'), Dish(name='carrot cake', ingredients=['flour', 'sugar', 'eggs', 'carrots', 'walnuts'], location='American')])

## State (Orchestration)

In LangGraph, **state** is the shared memory that flows through your workflow. It captures everything the agents or nodes need to know, modify, or pass along to the next step.

Think of it as the **context backpack** carried from node to node.

Unlike a traditional chain where only input and output are passed along, LangGraph gives each node access to a structured `state` dictionary, which can hold:

- The original user input or goal.
- Intermediate results (for example, parsed sections, completed plans).
- Final summaries or evaluations.
- Data for decision-making or routing (for example, risk level, retry count).

### Why State Matters in Agentic Workflows

Agentic systems often involve multiple steps, roles, and decision points. Having a shared, evolving state lets your agents:

- Access relevant context from earlier steps.
- Modify or enrich the state as they work.
- Route logic dynamically based on conditions.
- Loop, retry, or reflect based on what’s in the state.


In [11]:
from typing import TypedDict,Annotated
import operator

In [13]:
class State(TypedDict):
    
    meals:str
    sections:list[Dish]
    completed_menu: Annotated[list[str], operator.add]
    final_answer: str

In [14]:
initial_state:State = {
    "meals": "Spaghetti Bolognese and Chicken Stir Fry",
    "sections":[],
    "completed_menu": [],
    "final_answer": ""
}

In [16]:
initial_sections = planner_pipe.invoke({
    "meals": initial_state['meals']
})

In [21]:
for i, section in enumerate(initial_sections.sections):
    print(f"Dish {i+1}")
    
    initial_state['sections'].append(section)
    print(f"Dish name: {section.name}")
    print(f"Location/Cuisine: {section.location}")
    print(f"Ingredients: {',' .join(section.ingredients)}.\n")

Dish 1
Dish name: Spaghetti Bolognese
Location/Cuisine: Italian
Ingredients: spaghetti, ground beef, onion, carrot, celery, tomato paste, canned tomatoes, olive oil, salt, black pepper, basil.

Dish 2
Dish name: Chicken Stir Fry
Location/Cuisine: Chinese
Ingredients: chicken breast, vegetable oil, onion, bell pepper, broccoli, soy sauce, garlic, ginger, salt, black pepper.



## Orchestrator Node

The **orchestrator** is responsible for high-level planning and acts as the central coordinator in the LangGraph workflow. It takes a user's input and produces structured subtasks for other nodes to handle.

In our workflow, the orchestrator:

- Takes the raw input (for example, `"Spaghetti Bolognese and Chicken Stir Fry"`).
- Uses an LLM to break it down into structured `Dish` objects.
- Returns the result as a dict with the field `sections` for worker nodes to process.

This enables **fan-out parallelism**, where multiple workers can now act independently.
